# Create training programs from skill incidence

## 0. Setup

In [1]:
import pandas as pd
from pathlib import Path

# Data directories
DATA_DIR = Path("../data")
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

print(f"Silver data directory: {SILVER_DIR}")
print(f"Gold data directory: {GOLD_DIR}")

Silver data directory: ../data/silver
Gold data directory: ../data/gold


## 1. Decide on industry level (NACE section vs division)

### 1.01 Read in occupation level data

In [2]:
# Read occupation level data
occupation_df = pd.read_csv(GOLD_DIR / "occupation_level_data.csv")

print(f"Loaded {len(occupation_df)} occupation records")
print(f"Columns: {list(occupation_df.columns)}")
occupation_df.head()

Loaded 555 occupation records
Columns: ['project_id', 'esco_id', 'occupation_esco', 'esco_description', 'industry_cat_code', 'industry_cat_label', 'industry_division_code', 'industry_division_label', 'onet_job_zone', 'onet_job_zone_label', 'onet_job_zone_est', 'pad_job_titles', 'pad_activities', 'pad_skills', 'pad_quotes']


,project_id,esco_id,occupation_esco,esco_description,industry_cat_code,industry_cat_label,industry_division_code,industry_division_label,onet_job_zone,onet_job_zone_label,onet_job_zone_est,pad_job_titles,pad_activities,pad_skills,pad_quotes
0,P160708,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",70.0,70 Activities of head offices and management c...,2.0,2: Some Preparation Needed,minimum,"""procurement specialist""","""Manage procurement processes for project subc...","""public procurement regulations"", ""tender prep...","IV. PROJECT APPRAISAL SUMMARY: ""Procurement un..."
1,P160708,075e4d25-74b3-46fb-9b6e-6b53cdb196e7,credit analyst,Credit analysts investigate credit application...,L,L FINANCIAL AND INSURANCE ACTIVITIES,64.0,"64 Financial service activities, except insura...",2.0,2: Some Preparation Needed,minimum,"""credit analyst""","""Assess and identify commercial financial inst...","""financial analysis"", ""credit risk assessment""...","II. PROJECT DESCRIPTION: ""BOAD will identify C..."
2,P160708,0ba06640-e0ac-4911-9e43-289a8e41651e,corporate trainer,"Corporate trainers train, coach, and guide emp...",Q,Q EDUCATION,85.0,85 Education,4.0,4: Considerable Preparation Needed,minimum,"""financial sector training specialist"", ""train...","""Provide capacity building to commercial finan...","""training curriculum design"", ""financial produ...",ANNEX 13: CURRENT PRIVATE SECTOR PARTICIPATION...
3,P160708,0deaceea-69d8-4304-bc80-0b3d0491a241,field survey manager,Field survey managers organise and supervise i...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",73.0,"73 Activities of advertising, market research ...",5.0,5: Extensive Preparation Needed,unique,"""survey coordinator""","""Define sampling frames, select locations and ...","""sampling frame development"", ""cluster definit...","ANNEX 8: IMPACT EVALUATION: ""Households and en..."
4,P160708,109e0a5d-203d-4af6-8f70-692111335ec3,quality services manager,Quality services managers manage the quality o...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",70.0,70 Activities of head offices and management c...,4.0,4: Considerable Preparation Needed,unique,"""quality assurance specialist""","""Support adoption and implementation of a regi...","""quality assurance frameworks"", ""standards dev...","VI. RESULTS FRAMEWORK AND MONITORING: ""ECREEE ..."


### 1.02 Tabulate industry categories

In [3]:
# Count by industry category (NACE section)
industry_cat_counts = occupation_df['industry_cat_label'].value_counts().reset_index()
industry_cat_counts.columns = ['Industry Category (Section)', 'Count']

print(f"Industry Categories (NACE Sections): {len(industry_cat_counts)} unique categories\n")
print(industry_cat_counts)

print("\n" + "="*80 + "\n")

# Count by industry division
industry_div_counts = occupation_df['industry_division_label'].value_counts().reset_index()
industry_div_counts.columns = ['Industry Division', 'Count']

print(f"Industry Divisions: {len(industry_div_counts)} unique divisions\n")
print(industry_div_counts)

Industry Categories (NACE Sections): 17 unique categories

                          Industry Category (Section)  Count
0   N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...    248
1                                      F CONSTRUCTION     63
2   P PUBLIC ADMINISTRATION AND DEFENCE; COMPULSOR...     50
3   D ELECTRICITY, GAS, STEAM AND AIR CONDITIONING...     39
4   K TELECOMMUNICATION, COMPUTER PROGRAMMING, CON...     38
5                L FINANCIAL AND INSURANCE ACTIVITIES     31
6                                         Q EDUCATION     21
7     O ADMINISTRATIVE AND SUPPORT SERVICE ACTIVITIES     17
8                                     C MANUFACTURING     13
9           R HUMAN HEALTH AND SOCIAL WORK ACTIVITIES      9
10                       H TRANSPORTATION AND STORAGE      6
11                         T OTHER SERVICE ACTIVITIES      5
12                           M REAL ESTATE ACTIVITIES      5
13  E WATER SUPPLY; SEWERAGE, WASTE MANAGEMENT AND...      4
14                      S 

### 1.03 Merge on Project Information

In [4]:
# Read project level data
project_df = pd.read_csv(GOLD_DIR / "project_level_data.csv")

print(f"Loaded {len(project_df)} project records")
print(f"Columns: {list(project_df.columns)}")
print(f"\nSample data:")
project_df.head()

Loaded 11 project records
Columns: ['project_id', 'project_title', 'geography', 'short_summary', 'long_summary', 'closing_date', 'total_project_cost_1', 'status', 'team_leader', 'borrower_2', 'disclosure_date', 'effective_date', 'implementing_agency', 'region', 'environmental_and_social_risk', 'last_stage_reached', 'last_update_date']

Sample data:


,project_id,project_title,geography,short_summary,long_summary,closing_date,total_project_cost_1,status,team_leader,borrower_2,disclosure_date,effective_date,implementing_agency,region,environmental_and_social_risk,last_stage_reached,last_update_date
0,P119893,Ethiopia - Electricity Network Reinforcement a...,Ethiopia,The Electricity Network Reinforcement and Expa...,The Electricity Network Reinforcement and Expa...,"March 31, 2025",US$ 275.00 million,Closed,Abdulhakim Mohammed Abdisubhan,Federal Democratic Republic of Ethiopia,"December 22, 2011","January 4, 2013","Development Bank of Ethiopia,Ethiopia Electric...",Eastern and Southern Africa,Not Applicable,Bank Approved,"June 20, 2024"
1,P173506,"Congo, Democratic Republic of - Access Governa...",Democratic Republic of Congo,AGREE project expands renewable-based electric...,The AGREE project is a large water and energy ...,"September 30, 2029",US$ 939.00 million,Active,"Didier Makoso Tsasa , Fabrice Karl Bertholet, ...",DEMOCRATIC REPUBLIC OF CONGO,"January 6, 2021","May 18, 2023",Ministère des Ressources Hydrauliques et de l'...,Eastern and Southern Africa,High,Bank Approved,"June 17, 2024"
2,P176731,"Ethiopia - Power Sector Reform, Investment, an...",Ethiopia,This project supports Ethiopia's phased electr...,This project supports a phased electricity pro...,"August 30, 2030",US$ 537.00 million,Active,"Janina Franco , Abdulhakim Mohammed Abdisubhan...",Federal Democratic Republic of Ethiopia,"August 15, 2023","June 19, 2024","Ethiopia Electric Power,Ethiopia Electric Utility",Eastern and Southern Africa,High,Bank Approved,"June 16, 2023"
3,P507759,Mozambique - Accelerating Sustainable and Clea...,Mozambique,"ASCENT Mozambique finances on-grid expansion, ...",ASCENT Mozambique is Phase 10 of the ASCENT mu...,"December 31, 2030",US$ 0.00 million,Active,"Jenny Jing Chao , Maria Arango",Republic of Mozambique,"November 29, 2024","August 21, 2025","Electricidade de Moçambique (EDM),Fundo de Ene...",Eastern and Southern Africa,Substantial,Bank Approved,"March 22, 2025"
4,P180547,Eastern and Southern Africa - Accelerating Sus...,Eastern and Southern Africa (more than 20 Coun...,ASCENT accelerates sustainable and clean energ...,ASCENT is a regional program to accelerate sus...,"December 31, 2030",US$ 10000.00 million,Active,"Monali Ranade , Dana Rysankova, Alona Kazantseva",Common Market for Eastern and Southern Africa ...,"July 7, 2023","April 5, 2024",Common Market for Eastern and Southern Africa ...,Eastern and Southern Africa,Substantial,Bank Approved,"July 16, 2023"


In [5]:
# Merge project information onto occupation data
occupation_df = occupation_df.merge(
    project_df[['project_id', 'project_title', 'short_summary']], 
    on='project_id', 
    how='left'
)

print(f"Merged project information onto occupation_df")
print(f"New columns: project_title, short_summary")
print(f"Shape: {occupation_df.shape}")
print(f"\nSample merged data:")
occupation_df[['project_id', 'project_title', 'short_summary', 'occupation_esco']].head()

Merged project information onto occupation_df
New columns: project_title, short_summary
Shape: (555, 17)

Sample merged data:


,project_id,project_title,short_summary,occupation_esco
0,P160708,Western Africa - Regional Off-Grid Electrifica...,This regional project scales modern stand-alon...,procurement category specialist
1,P160708,Western Africa - Regional Off-Grid Electrifica...,This regional project scales modern stand-alon...,credit analyst
2,P160708,Western Africa - Regional Off-Grid Electrifica...,This regional project scales modern stand-alon...,corporate trainer
3,P160708,Western Africa - Regional Off-Grid Electrifica...,This regional project scales modern stand-alon...,field survey manager
4,P160708,Western Africa - Regional Off-Grid Electrifica...,This regional project scales modern stand-alon...,quality services manager


## 2. Skill counts by division and job zone

### 2.01 Read in skill level file

In [6]:
# Read skill level data
skill_df = pd.read_csv(GOLD_DIR / "skill_level_data.csv")

print(f"Loaded {len(skill_df)} skill records")
print(f"Columns: {list(skill_df.columns)}")
skill_df.head()

Loaded 10950 skill records
Columns: ['project_id', 'esco_id', 'skill_code', 'skill_label', 'skill_type', 'skill_category_label', 'top_five']


,project_id,esco_id,skill_code,skill_label,skill_type,skill_category_label,top_five
0,P507759,0561328b-875b-4ae2-9ba1-9af9049aef01,23ac233d-84ad-4517-b0f5-8ca19ba2614e,monitor developments in field of expertise,skill/competence,information skills,False
1,P507759,0561328b-875b-4ae2-9ba1-9af9049aef01,31599ac3-e03c-42f8-8f08-b549af8234f2,draft procurement technical specifications,skill/competence,management skills,True
2,P507759,0561328b-875b-4ae2-9ba1-9af9049aef01,3bc08c26-2f98-4730-bbb6-ec4a8566e3cf,procurement lifecycle,knowledge,"business, administration and law",True
3,P507759,0561328b-875b-4ae2-9ba1-9af9049aef01,3cd35f5d-ce6d-4f14-9a09-53d7a28d834c,maintain relationship with suppliers,skill/competence,"communication, collaboration and creativity",False
4,P507759,0561328b-875b-4ae2-9ba1-9af9049aef01,5592ab32-4e7a-4cda-8e64-ca36d5de8a10,adapt to changing situations,skill/competence,working with computers,False


### 2.02 merge on occupations df

In [7]:
# Merge skill data with occupation data on esco_id
merged_df = occupation_df.merge(skill_df, on='esco_id', how='inner', suffixes=('', '_skill'))

print(f"Merged dataset: {len(merged_df)} records (occupation × skill pairs)")
print(f"Columns: {list(merged_df.columns)}")
print(f"\nSample merged data:")
merged_df.head()

Merged dataset: 56319 records (occupation × skill pairs)
Columns: ['project_id', 'esco_id', 'occupation_esco', 'esco_description', 'industry_cat_code', 'industry_cat_label', 'industry_division_code', 'industry_division_label', 'onet_job_zone', 'onet_job_zone_label', 'onet_job_zone_est', 'pad_job_titles', 'pad_activities', 'pad_skills', 'pad_quotes', 'project_title', 'short_summary', 'project_id_skill', 'skill_code', 'skill_label', 'skill_type', 'skill_category_label', 'top_five']

Sample merged data:


,project_id,esco_id,occupation_esco,esco_description,industry_cat_code,industry_cat_label,industry_division_code,industry_division_label,onet_job_zone,onet_job_zone_label,...,pad_skills,pad_quotes,project_title,short_summary,project_id_skill,skill_code,skill_label,skill_type,skill_category_label,top_five
0,P160708,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",70.0,70 Activities of head offices and management c...,2.0,2: Some Preparation Needed,...,"""public procurement regulations"", ""tender prep...","IV. PROJECT APPRAISAL SUMMARY: ""Procurement un...",Western Africa - Regional Off-Grid Electrifica...,This regional project scales modern stand-alon...,P507759,23ac233d-84ad-4517-b0f5-8ca19ba2614e,monitor developments in field of expertise,skill/competence,information skills,False
1,P160708,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",70.0,70 Activities of head offices and management c...,2.0,2: Some Preparation Needed,...,"""public procurement regulations"", ""tender prep...","IV. PROJECT APPRAISAL SUMMARY: ""Procurement un...",Western Africa - Regional Off-Grid Electrifica...,This regional project scales modern stand-alon...,P507759,31599ac3-e03c-42f8-8f08-b549af8234f2,draft procurement technical specifications,skill/competence,management skills,True
2,P160708,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",70.0,70 Activities of head offices and management c...,2.0,2: Some Preparation Needed,...,"""public procurement regulations"", ""tender prep...","IV. PROJECT APPRAISAL SUMMARY: ""Procurement un...",Western Africa - Regional Off-Grid Electrifica...,This regional project scales modern stand-alon...,P507759,3bc08c26-2f98-4730-bbb6-ec4a8566e3cf,procurement lifecycle,knowledge,"business, administration and law",True
3,P160708,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",70.0,70 Activities of head offices and management c...,2.0,2: Some Preparation Needed,...,"""public procurement regulations"", ""tender prep...","IV. PROJECT APPRAISAL SUMMARY: ""Procurement un...",Western Africa - Regional Off-Grid Electrifica...,This regional project scales modern stand-alon...,P507759,3cd35f5d-ce6d-4f14-9a09-53d7a28d834c,maintain relationship with suppliers,skill/competence,"communication, collaboration and creativity",False
4,P160708,0561328b-875b-4ae2-9ba1-9af9049aef01,procurement category specialist,Procurement category specialists are experts i...,N,"N PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIV...",70.0,70 Activities of head offices and management c...,2.0,2: Some Preparation Needed,...,"""public procurement regulations"", ""tender prep...","IV. PROJECT APPRAISAL SUMMARY: ""Procurement un...",Western Africa - Regional Off-Grid Electrifica...,This regional project scales modern stand-alon...,P507759,5592ab32-4e7a-4cda-8e64-ca36d5de8a10,adapt to changing situations,skill/competence,working with computers,False


### 2.03 Create combined job zone categories

In [8]:
# Create combined job zone categories
# Map job zones: 1-2 → "1-2", 3 → "3", 4-5 → "4-5"
def map_job_zone(zone):
    if pd.isna(zone):
        return None
    if zone in [1.0, 2.0]:
        return "1-2"
    elif zone == 3.0:
        return "3"
    elif zone in [4.0, 5.0]:
        return "4-5"
    else:
        return None

merged_df['combined_job_zone'] = merged_df['onet_job_zone'].apply(map_job_zone)

print(f"Combined job zone categories created")
print(f"\nValue counts:")
print(merged_df['combined_job_zone'].value_counts())
print(f"\nSample data:")
merged_df[['occupation_esco', 'onet_job_zone', 'onet_job_zone_label', 'combined_job_zone']].head(10)

Combined job zone categories created

Value counts:
combined_job_zone
4-5    39319
1-2    10890
3       6109
Name: count, dtype: int64

Sample data:


,occupation_esco,onet_job_zone,onet_job_zone_label,combined_job_zone
0,procurement category specialist,2.0,2: Some Preparation Needed,1-2
1,procurement category specialist,2.0,2: Some Preparation Needed,1-2
2,procurement category specialist,2.0,2: Some Preparation Needed,1-2
3,procurement category specialist,2.0,2: Some Preparation Needed,1-2
4,procurement category specialist,2.0,2: Some Preparation Needed,1-2
5,procurement category specialist,2.0,2: Some Preparation Needed,1-2
6,procurement category specialist,2.0,2: Some Preparation Needed,1-2
7,procurement category specialist,2.0,2: Some Preparation Needed,1-2
8,procurement category specialist,2.0,2: Some Preparation Needed,1-2
9,procurement category specialist,2.0,2: Some Preparation Needed,1-2


### 2.04 Skill counts by division and job zone

In [9]:
# Group by division, job zone, and skill attributes
# Create separate columns for each unique occupation and project

# First, create the base aggregation with occupation count
skill_incidence_df = (
    merged_df.groupby(
        ['industry_cat_label', 'industry_division_label', 'combined_job_zone', 'skill_code', 'skill_label', 'skill_type', 'top_five', 'skill_category_label'],
        dropna=False
    )
    .agg(occupation_count=('esco_id', 'nunique'))
    .reset_index()
)

# Now add occupation and project columns dynamically
def add_occupation_project_columns(row):
    # Filter merged_df for this specific group
    mask = (
        (merged_df['industry_cat_label'] == row['industry_cat_label']) &
        (merged_df['industry_division_label'] == row['industry_division_label']) &
        (merged_df['combined_job_zone'] == row['combined_job_zone']) &
        (merged_df['skill_code'] == row['skill_code'])
    )
    group_data = merged_df[mask]
    
    # Get unique occupations sorted by esco_id
    occ_data = group_data[['esco_id', 'occupation_esco', 'esco_description']].drop_duplicates().sort_values('esco_id')
    
    # Get unique projects sorted by project_id
    proj_data = group_data[['project_id', 'project_title', 'short_summary']].drop_duplicates().sort_values('project_id')
    
    # Add occupation columns
    for idx, (_, occ_row) in enumerate(occ_data.iterrows(), start=1):
        row[f'occupation_lbl_{idx:02d}'] = occ_row['occupation_esco']
        row[f'occupation_esco_id_{idx:02d}'] = occ_row['esco_id']
        row[f'occupation_desc_{idx:02d}'] = occ_row['esco_description']
    
    # Add project columns
    for idx, (_, proj_row) in enumerate(proj_data.iterrows(), start=1):
        row[f'project_id_{idx:02d}'] = proj_row['project_id']
        row[f'project_title_{idx:02d}'] = proj_row['project_title']
        row[f'project_short_summary_{idx:02d}'] = proj_row['short_summary']
    
    return row

print("Creating skill incidence dataframe with occupation and project columns...")
skill_incidence_df = skill_incidence_df.apply(add_occupation_project_columns, axis=1)

print(f"\nSkill incidence dataframe created: {len(skill_incidence_df)} unique skill-division-jobzone combinations")
print(f"\nColumns: {list(skill_incidence_df.columns)}")
print(f"\nSample data:")
skill_incidence_df.head(10)

Creating skill incidence dataframe with occupation and project columns...

Skill incidence dataframe created: 4129 unique skill-division-jobzone combinations

Columns: ['combined_job_zone', 'industry_cat_label', 'industry_division_label', 'occupation_count', 'occupation_desc_01', 'occupation_desc_02', 'occupation_desc_03', 'occupation_desc_04', 'occupation_desc_05', 'occupation_desc_06', 'occupation_desc_07', 'occupation_desc_08', 'occupation_desc_09', 'occupation_desc_10', 'occupation_desc_11', 'occupation_desc_12', 'occupation_desc_13', 'occupation_esco_id_01', 'occupation_esco_id_02', 'occupation_esco_id_03', 'occupation_esco_id_04', 'occupation_esco_id_05', 'occupation_esco_id_06', 'occupation_esco_id_07', 'occupation_esco_id_08', 'occupation_esco_id_09', 'occupation_esco_id_10', 'occupation_esco_id_11', 'occupation_esco_id_12', 'occupation_esco_id_13', 'occupation_lbl_01', 'occupation_lbl_02', 'occupation_lbl_03', 'occupation_lbl_04', 'occupation_lbl_05', 'occupation_lbl_06', 'occ

,combined_job_zone,industry_cat_label,industry_division_label,occupation_count,occupation_desc_01,occupation_desc_02,occupation_desc_03,occupation_desc_04,occupation_desc_05,occupation_desc_06,...,project_title_07,project_title_08,project_title_09,project_title_10,project_title_11,skill_category_label,skill_code,skill_label,skill_type,top_five
0,1-2,B MINING AND QUARRYING,09 Mining support service activities,1,Drill operators supervise a team during riggin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,working with machinery and specialised equipment,11dc8e6b-dffa-42e4-9de1-c47293a848ff,operate pumping equipment,skill/competence,False
1,1-2,B MINING AND QUARRYING,09 Mining support service activities,1,Drill operators supervise a team during riggin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,assisting and caring,156f8c5b-894a-4ccc-a70e-37a2726f3f00,work ergonomically,skill/competence,False
2,1-2,B MINING AND QUARRYING,09 Mining support service activities,1,Drill operators supervise a team during riggin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,information skills,3a560dc4-ee2e-4437-b94f-c076a2b5f25e,inspect water wells,skill/competence,False
3,1-2,B MINING AND QUARRYING,09 Mining support service activities,1,Drill operators supervise a team during riggin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,"engineering, manufacturing and construction",4064716d-6714-48ea-95a0-ef81c2b31672,mine safety legislation,knowledge,True
4,1-2,B MINING AND QUARRYING,09 Mining support service activities,1,Drill operators supervise a team during riggin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,information skills,43b13058-68d0-40c6-8031-caf5a7b7d20c,keep task records,skill/competence,True
5,1-2,B MINING AND QUARRYING,09 Mining support service activities,1,Drill operators supervise a team during riggin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,assisting and caring,6badc79b-4353-4f95-b248-57a9ed07c7ac,supervise worker safety,skill/competence,True
6,1-2,B MINING AND QUARRYING,09 Mining support service activities,1,Drill operators supervise a team during riggin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,management skills,85aa1871-db96-42f7-801f-c39f924de2d7,plan shifts of employees,skill/competence,False
7,1-2,B MINING AND QUARRYING,09 Mining support service activities,1,Drill operators supervise a team during riggin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,management skills,8a98707e-b2ee-48fe-9f36-e47e3555ecf7,evaluate employees work,skill/competence,False
8,1-2,B MINING AND QUARRYING,09 Mining support service activities,1,Drill operators supervise a team during riggin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,working with machinery and specialised equipment,91713ef2-3d35-4df8-92a6-2fad76c60681,install oil rig,skill/competence,False
9,1-2,B MINING AND QUARRYING,09 Mining support service activities,1,Drill operators supervise a team during riggin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,"communication, collaboration and creativity",918f5211-b99f-4b29-9a34-6e475583ef55,write work-related reports,skill/competence,False


### 2.05 Move occupation and project information to the end

In [10]:
# Identify all column groups
base_cols = ['industry_cat_label', 'industry_division_label', 'combined_job_zone', 
             'skill_code', 'skill_label', 'skill_type', 'skill_category_label', 'top_five', 'occupation_count']

# Get all numbered suffixes present in the dataframe
project_ids = [col for col in skill_incidence_df.columns if col.startswith('project_id_')]
occupation_ids = [col for col in skill_incidence_df.columns if col.startswith('occupation_esco_id_')]

# Extract unique numbers from both sets
project_nums = sorted(set([col.split('_')[-1] for col in project_ids]))
occupation_nums = sorted(set([col.split('_')[-1] for col in occupation_ids]))

# Get the union of all numbers
all_nums = sorted(set(project_nums + occupation_nums))

# Build the ordered column list
ordered_cols = base_cols.copy()

# For each number, add the full set of columns in order
for num in all_nums:
    for prefix in ['project_id', 'project_title', 'project_short_summary', 
                   'occupation_esco_id', 'occupation_lbl', 'occupation_desc']:
        col_name = f'{prefix}_{num}'
        if col_name in skill_incidence_df.columns:
            ordered_cols.append(col_name)

# Reorder the dataframe
skill_incidence_df = skill_incidence_df[ordered_cols]

print(f"Reordered columns - first 10: {list(skill_incidence_df.columns[:10])}")
print(f"Last 10: {list(skill_incidence_df.columns[-10:])}")
print(f"\nTotal columns: {len(skill_incidence_df.columns)}")
print(f"\nSample data (first few rows, first few columns):")
skill_incidence_df.iloc[:3, :15]

Reordered columns - first 10: ['industry_cat_label', 'industry_division_label', 'combined_job_zone', 'skill_code', 'skill_label', 'skill_type', 'skill_category_label', 'top_five', 'occupation_count', 'project_id_01']
Last 10: ['project_short_summary_11', 'occupation_esco_id_11', 'occupation_lbl_11', 'occupation_desc_11', 'occupation_esco_id_12', 'occupation_lbl_12', 'occupation_desc_12', 'occupation_esco_id_13', 'occupation_lbl_13', 'occupation_desc_13']

Total columns: 81

Sample data (first few rows, first few columns):


,industry_cat_label,industry_division_label,combined_job_zone,skill_code,skill_label,skill_type,skill_category_label,top_five,occupation_count,project_id_01,project_title_01,project_short_summary_01,occupation_esco_id_01,occupation_lbl_01,occupation_desc_01
0,B MINING AND QUARRYING,09 Mining support service activities,1-2,11dc8e6b-dffa-42e4-9de1-c47293a848ff,operate pumping equipment,skill/competence,working with machinery and specialised equipment,False,1,P176731,"Ethiopia - Power Sector Reform, Investment, an...",This project supports Ethiopia's phased electr...,622b1d5e-bf14-4ec0-b83e-3bfcf8dff624,drill operator,Drill operators supervise a team during riggin...
1,B MINING AND QUARRYING,09 Mining support service activities,1-2,156f8c5b-894a-4ccc-a70e-37a2726f3f00,work ergonomically,skill/competence,assisting and caring,False,1,P176731,"Ethiopia - Power Sector Reform, Investment, an...",This project supports Ethiopia's phased electr...,622b1d5e-bf14-4ec0-b83e-3bfcf8dff624,drill operator,Drill operators supervise a team during riggin...
2,B MINING AND QUARRYING,09 Mining support service activities,1-2,3a560dc4-ee2e-4437-b94f-c076a2b5f25e,inspect water wells,skill/competence,information skills,False,1,P176731,"Ethiopia - Power Sector Reform, Investment, an...",This project supports Ethiopia's phased electr...,622b1d5e-bf14-4ec0-b83e-3bfcf8dff624,drill operator,Drill operators supervise a team during riggin...


In [11]:
skill_incidence_df.head(200)

,industry_cat_label,industry_division_label,combined_job_zone,skill_code,skill_label,skill_type,skill_category_label,top_five,occupation_count,project_id_01,...,project_short_summary_11,occupation_esco_id_11,occupation_lbl_11,occupation_desc_11,occupation_esco_id_12,occupation_lbl_12,occupation_desc_12,occupation_esco_id_13,occupation_lbl_13,occupation_desc_13
0,B MINING AND QUARRYING,09 Mining support service activities,1-2,11dc8e6b-dffa-42e4-9de1-c47293a848ff,operate pumping equipment,skill/competence,working with machinery and specialised equipment,False,1,P176731,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B MINING AND QUARRYING,09 Mining support service activities,1-2,156f8c5b-894a-4ccc-a70e-37a2726f3f00,work ergonomically,skill/competence,assisting and caring,False,1,P176731,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B MINING AND QUARRYING,09 Mining support service activities,1-2,3a560dc4-ee2e-4437-b94f-c076a2b5f25e,inspect water wells,skill/competence,information skills,False,1,P176731,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,B MINING AND QUARRYING,09 Mining support service activities,1-2,4064716d-6714-48ea-95a0-ef81c2b31672,mine safety legislation,knowledge,"engineering, manufacturing and construction",True,1,P176731,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,B MINING AND QUARRYING,09 Mining support service activities,1-2,43b13058-68d0-40c6-8031-caf5a7b7d20c,keep task records,skill/competence,information skills,True,1,P176731,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,C MANUFACTURING,"33 Repair, maintenance and installation of mac...",1-2,7f67f82e-73c5-4dc0-b506-8e0b1a84f614,bind wire,skill/competence,handling and moving,False,1,P511453,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196,C MANUFACTURING,"33 Repair, maintenance and installation of mac...",1-2,86df7af2-f9f3-4c06-a500-7f1fba9e78fe,follow health and safety procedures in constru...,skill/competence,assisting and caring,True,1,P511453,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197,C MANUFACTURING,"33 Repair, maintenance and installation of mac...",1-2,a99c6783-50c0-4f7a-99df-bf93429ef6dc,use precision tools,skill/competence,handling and moving,False,1,P511453,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,C MANUFACTURING,"33 Repair, maintenance and installation of mac...",1-2,b92758d4-4591-4df1-9ceb-f907aa71ebd9,use measurement instruments,skill/competence,information skills,False,1,P511453,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2.05 Sort and Create IDs

In [12]:
# Sort by industry category, division, job zone, top_five (True first), and skill code
skill_incidence_df = skill_incidence_df.sort_values(
    by=['industry_cat_label', 'industry_division_label', 'combined_job_zone', 'top_five', 'skill_category_label', 'skill_code'],
    ascending=[True, True, True, False, True, True]  # top_five descending so True comes first
).reset_index(drop=True)

# Create skill_bundle_id: 3-digit identifier for each industry_division_label + combined_job_zone combination
bundle_groups = skill_incidence_df.groupby(['industry_division_label', 'combined_job_zone'], dropna=False).ngroup()
skill_incidence_df['skill_bundle_id'] = (bundle_groups + 1).apply(lambda x: f"{x:03d}")

# Create order_within_bundle: skill order within each bundle
skill_incidence_df['order_within_bundle'] = skill_incidence_df.groupby(
    ['industry_division_label', 'combined_job_zone'], dropna=False
).cumcount() + 1

# Create skill_number: skill_bundle_id + 3-digit order within bundle
skill_incidence_df['skill_number'] = (
    skill_incidence_df['skill_bundle_id'] + 
    skill_incidence_df['order_within_bundle'].apply(lambda x: f"{x:03d}")
)

# Reorder columns to put IDs first
cols = skill_incidence_df.columns.tolist()
id_cols = ['skill_bundle_id', 'order_within_bundle']
other_cols = [col for col in cols if col not in id_cols]
skill_incidence_df = skill_incidence_df[id_cols + other_cols]

print(f"Sorted and added IDs")
print(f"\nColumns: {list(skill_incidence_df.columns)}")
print(f"\nSample data:")
skill_incidence_df[['skill_bundle_id', 'order_within_bundle', 'skill_number', 'industry_division_label', 'combined_job_zone', 'top_five', 'skill_code', 'skill_label']].head(20)

Sorted and added IDs

Columns: ['skill_bundle_id', 'order_within_bundle', 'industry_cat_label', 'industry_division_label', 'combined_job_zone', 'skill_code', 'skill_label', 'skill_type', 'skill_category_label', 'top_five', 'occupation_count', 'project_id_01', 'project_title_01', 'project_short_summary_01', 'occupation_esco_id_01', 'occupation_lbl_01', 'occupation_desc_01', 'project_id_02', 'project_title_02', 'project_short_summary_02', 'occupation_esco_id_02', 'occupation_lbl_02', 'occupation_desc_02', 'project_id_03', 'project_title_03', 'project_short_summary_03', 'occupation_esco_id_03', 'occupation_lbl_03', 'occupation_desc_03', 'project_id_04', 'project_title_04', 'project_short_summary_04', 'occupation_esco_id_04', 'occupation_lbl_04', 'occupation_desc_04', 'project_id_05', 'project_title_05', 'project_short_summary_05', 'occupation_esco_id_05', 'occupation_lbl_05', 'occupation_desc_05', 'project_id_06', 'project_title_06', 'project_short_summary_06', 'occupation_esco_id_06', 'o

,skill_bundle_id,order_within_bundle,skill_number,industry_division_label,combined_job_zone,top_five,skill_code,skill_label
0,001,1,001001,09 Mining support service activities,1-2,True,6badc79b-4353-4f95-b248-57a9ed07c7ac,supervise worker safety
1,001,2,001002,09 Mining support service activities,1-2,True,4064716d-6714-48ea-95a0-ef81c2b31672,mine safety legislation
2,001,3,001003,09 Mining support service activities,1-2,True,efa141df-f382-418f-9121-bd88fd735669,mechanics
3,001,4,001004,09 Mining support service activities,1-2,True,db99d662-4230-49b3-a9e9-b0043a70c3c3,operate drilling equipment
4,001,5,001005,09 Mining support service activities,1-2,True,43b13058-68d0-40c6-8031-caf5a7b7d20c,keep task records
5,001,6,001006,09 Mining support service activities,1-2,False,156f8c5b-894a-4ccc-a70e-37a2726f3f00,work ergonomically
6,001,7,001007,09 Mining support service activities,1-2,False,918f5211-b99f-4b29-9a34-6e475583ef55,write work-related reports
7,001,8,001008,09 Mining support service activities,1-2,False,d0f2f8a7-d935-4a6d-8e54-dcc3e5e2cbeb,present reports
8,001,9,001009,09 Mining support service activities,1-2,False,f14ff4b7-be1e-4b55-b39b-520005f8a97e,liaise with managers
9,001,10,001010,09 Mining support service activities,1-2,False,3a560dc4-ee2e-4437-b94f-c076a2b5f25e,inspect water wells


In [13]:
skill_incidence_df.head(100)

,skill_bundle_id,order_within_bundle,industry_cat_label,industry_division_label,combined_job_zone,skill_code,skill_label,skill_type,skill_category_label,top_five,...,occupation_esco_id_11,occupation_lbl_11,occupation_desc_11,occupation_esco_id_12,occupation_lbl_12,occupation_desc_12,occupation_esco_id_13,occupation_lbl_13,occupation_desc_13,skill_number
0,001,1,B MINING AND QUARRYING,09 Mining support service activities,1-2,6badc79b-4353-4f95-b248-57a9ed07c7ac,supervise worker safety,skill/competence,assisting and caring,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,001001
1,001,2,B MINING AND QUARRYING,09 Mining support service activities,1-2,4064716d-6714-48ea-95a0-ef81c2b31672,mine safety legislation,knowledge,"engineering, manufacturing and construction",True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,001002
2,001,3,B MINING AND QUARRYING,09 Mining support service activities,1-2,efa141df-f382-418f-9121-bd88fd735669,mechanics,knowledge,"engineering, manufacturing and construction",True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,001003
3,001,4,B MINING AND QUARRYING,09 Mining support service activities,1-2,db99d662-4230-49b3-a9e9-b0043a70c3c3,operate drilling equipment,skill/competence,handling and moving,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,001004
4,001,5,B MINING AND QUARRYING,09 Mining support service activities,1-2,43b13058-68d0-40c6-8031-caf5a7b7d20c,keep task records,skill/competence,information skills,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,001005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,005,6,C MANUFACTURING,27 Manufacture of electrical equipment,1-2,ed291aa0-8b2f-4179-b2f7-540ea6f63bf3,electrical wiring diagrams,knowledge,"engineering, manufacturing and construction",True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,005006
96,005,7,C MANUFACTURING,27 Manufacture of electrical equipment,1-2,b07daddc-8625-4360-946e-ad2b0e56ebf6,read engineering drawings,skill/competence,information skills,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,005007
97,005,8,C MANUFACTURING,27 Manufacture of electrical equipment,1-2,6122d586-5978-431f-8e7a-96e61fc1f3fc,wear appropriate protective gear,skill/competence,assisting and caring,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,005008
98,005,9,C MANUFACTURING,27 Manufacture of electrical equipment,1-2,42b23922-1c40-4dbe-9e0c-7a568dfdf06b,adjust engineering designs,skill/competence,"communication, collaboration and creativity",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,005009


### 2.05 Save to Gold for analysis


In [14]:
# Save the skill incidence dataframe to gold directory as training_program_bundles.csv
output_path = GOLD_DIR / "training_program_bundles.csv"
skill_incidence_df.to_csv(output_path, index=False)
print(f"Saved skill incidence dataframe to {output_path}")

Saved skill incidence dataframe to ../data/gold/training_program_bundles.csv
